# ???????? EfficientDet ??? ???????? ???????? ?? ???

??????? ??? Google Colab. ????????? ???????, ????????? ???????? EfficientDet ? ????????? ???????? ????????? ?? Google Drive.


In [ ]:
!pip install -q kagglehub torchmetrics pycocotools effdet timm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

PROJECT_NAME = 'brain-mri-efficientdet'
WORKDIR = Path('/content') / PROJECT_NAME
ARTIFACTS_DIR = Path('/content/drive/MyDrive') / PROJECT_NAME
DATASET_DIR = WORKDIR / 'dataset'
PREDICTIONS_DIR = ARTIFACTS_DIR / 'predictions'
RUNS_DIR = ARTIFACTS_DIR / 'runs'

EPOCHS = 4
BATCH_SIZE = 4
NUM_WORKERS = 2
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
SEED = 42
SCORE_THRESHOLD = 0.35
TRAIN_FRACTION = 0.35
VAL_FRACTION = 0.5
IMAGE_SIZE = 512

CLASS_NAMES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
NUM_CLASSES = len(CLASS_NAMES)

WORKDIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print('WORKDIR:', WORKDIR)
print('ARTIFACTS_DIR:', ARTIFACTS_DIR)


In [ ]:
import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)


In [ ]:
import kagglehub
import shutil
from pathlib import Path

path = kagglehub.dataset_download('ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes')
dataset_path = Path(path)
print('Raw kagglehub path:', dataset_path)

candidate_roots = [dataset_path] + [p for p in dataset_path.rglob('*') if p.is_dir()]
resolved_root = None
for candidate in candidate_roots:
    if (candidate / 'Train').exists() and (candidate / 'Val').exists():
        resolved_root = candidate
        break

if resolved_root is None:
    raise FileNotFoundError('?? ??????? ????? ????? Train/Val ?????? ????????.')

dataset_target = DATASET_DIR / 'brain-tumor-mri'
if dataset_target.exists():
    shutil.rmtree(dataset_target)
shutil.copytree(resolved_root, dataset_target)

print('Resolved dataset root:', resolved_root)
print('Copied dataset to:', dataset_target)


In [ ]:
from pathlib import Path
import random

import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.transforms import functional as F

class BrainMRIDetectionDataset(Dataset):
    def __init__(self, dataset_root, split_name, class_names, image_size=512, drop_empty=True):
        self.dataset_root = Path(dataset_root)
        self.split_name = split_name
        self.class_names = class_names
        self.image_size = image_size
        self.drop_empty = drop_empty
        self.records = []
        split_dir = self.dataset_root / split_name

        skipped_missing = 0
        skipped_empty = 0
        skipped_invalid = 0

        for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
            for image_path in sorted((class_dir / 'images').glob('*')):
                label_path = class_dir / 'labels' / f'{image_path.stem}.txt'
                if not label_path.exists():
                    skipped_missing += 1
                    if drop_empty:
                        continue

                valid_boxes = 0
                if label_path.exists():
                    lines = label_path.read_text(encoding='utf-8').strip().splitlines()
                    if not lines:
                        skipped_empty += 1
                        if drop_empty:
                            continue
                    for line in lines:
                        if not line.strip():
                            continue
                        parts = line.split()
                        if len(parts) != 5:
                            continue
                        try:
                            class_id, x_center, y_center, box_width, box_height = map(float, parts)
                        except ValueError:
                            continue
                        if box_width <= 0 or box_height <= 0:
                            continue
                        if not (0 <= x_center <= 1 and 0 <= y_center <= 1):
                            continue
                        if not (0 < box_width <= 1 and 0 < box_height <= 1):
                            continue
                        valid_boxes += 1

                if drop_empty and valid_boxes == 0:
                    skipped_invalid += 1
                    continue

                self.records.append({
                    'image_path': image_path,
                    'label_path': label_path,
                })

        print(f'{split_name}: kept={len(self.records)}, skipped_missing={skipped_missing}, skipped_empty={skipped_empty}, skipped_invalid={skipped_invalid}')

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        image = Image.open(record['image_path']).convert('RGB')
        orig_w, orig_h = image.size
        image = image.resize((self.image_size, self.image_size))
        image_tensor = F.to_tensor(image)

        boxes = []
        labels = []

        if record['label_path'].exists():
            content = record['label_path'].read_text(encoding='utf-8').strip().splitlines()
            for line in content:
                if not line.strip():
                    continue
                parts = line.split()
                if len(parts) != 5:
                    continue
                class_id, x_center, y_center, box_width, box_height = map(float, parts)
                if box_width <= 0 or box_height <= 0:
                    continue

                x1 = (x_center - box_width / 2.0) * self.image_size
                y1 = (y_center - box_height / 2.0) * self.image_size
                x2 = (x_center + box_width / 2.0) * self.image_size
                y2 = (y_center + box_height / 2.0) * self.image_size

                x1 = max(0.0, min(x1, self.image_size - 1))
                y1 = max(0.0, min(y1, self.image_size - 1))
                x2 = max(0.0, min(x2, self.image_size - 1))
                y2 = max(0.0, min(y2, self.image_size - 1))

                if x2 <= x1 or y2 <= y1:
                    continue

                boxes.append([x1, y1, x2, y2])
                labels.append(int(class_id) + 1)

        if boxes:
            boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
            labels_tensor = torch.tensor(labels, dtype=torch.int64)
        else:
            boxes_tensor = torch.zeros((0, 4), dtype=torch.float32)
            labels_tensor = torch.zeros((0,), dtype=torch.int64)

        target = {
            'bbox': boxes_tensor,
            'cls': labels_tensor,
            'img_size': torch.tensor([self.image_size, self.image_size], dtype=torch.float32),
            'img_scale': torch.tensor([1.0], dtype=torch.float32),
            'orig_size': torch.tensor([orig_h, orig_w], dtype=torch.float32),
        }

        return image_tensor, target


def collate_fn(batch):
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets


def make_subset(dataset, fraction, seed=42):
    if fraction >= 1.0:
        return dataset
    size = len(dataset)
    subset_size = max(1, int(size * fraction))
    rng = random.Random(seed)
    indices = list(range(size))
    rng.shuffle(indices)
    return Subset(dataset, indices[:subset_size])

DATA_ROOT = DATASET_DIR / 'brain-tumor-mri'

full_train_dataset = BrainMRIDetectionDataset(DATA_ROOT, 'Train', CLASS_NAMES, image_size=IMAGE_SIZE, drop_empty=True)
full_val_dataset = BrainMRIDetectionDataset(DATA_ROOT, 'Val', CLASS_NAMES, image_size=IMAGE_SIZE, drop_empty=True)

train_dataset = make_subset(full_train_dataset, TRAIN_FRACTION, SEED)
val_dataset = make_subset(full_val_dataset, VAL_FRACTION, SEED)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)

print('Full train samples:', len(full_train_dataset))
print('Full val samples:', len(full_val_dataset))
print('Used train samples:', len(train_dataset))
print('Used val samples:', len(val_dataset))


In [ ]:
import json

dataset_summary = {
    'full_train_samples': len(full_train_dataset),
    'full_val_samples': len(full_val_dataset),
    'used_train_samples': len(train_dataset),
    'used_val_samples': len(val_dataset),
    'train_fraction': TRAIN_FRACTION,
    'val_fraction': VAL_FRACTION,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'image_size': IMAGE_SIZE,
}

summary_path = ARTIFACTS_DIR / 'dataset_summary.json'
summary_path.write_text(json.dumps(dataset_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(dataset_summary, ensure_ascii=False, indent=2))


In [ ]:
from effdet import create_model

model = create_model('tf_efficientdet_d0', bench_task='train', num_classes=NUM_CLASSES)
model.to(DEVICE)
model


In [ ]:
import pandas as pd
import torch
from tqdm.auto import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1)

history = []
best_map50 = -1.0
best_weights_path = RUNS_DIR / 'best_efficientdet.pt'
last_weights_path = RUNS_DIR / 'last_efficientdet.pt'


def move_targets_to_device(targets, device):
    moved = []
    for target in targets:
        moved.append({k: v.to(device) for k, v in target.items()})
    return moved


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0
    valid_steps = 0

    for images, targets in tqdm(loader, desc='Train', leave=False):
        images = images.to(device)
        targets = move_targets_to_device(targets, device)

        outputs = model(images, targets)
        loss = outputs['loss'] if isinstance(outputs, dict) else outputs.loss

        if not torch.isfinite(loss):
            print('Non-finite loss detected, skipping step')
            continue

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        valid_steps += 1

    return running_loss / max(valid_steps, 1)


def evaluate_model(model, loader, device, score_threshold=0.35):
    metric = MeanAveragePrecision(class_metrics=False)
    model.eval()

    with torch.no_grad():
        for images, targets in tqdm(loader, desc='Val', leave=False):
            images = images.to(device)
            batch_targets = targets
            outputs = model(images)

            if isinstance(outputs, dict):
                detections = outputs.get('detections', None)
            else:
                detections = getattr(outputs, 'detections', None)

            if detections is None:
                continue

            preds = []
            refs = []

            for det, target in zip(detections, batch_targets):
                det = det.detach().cpu()
                keep = det[:, 4] >= score_threshold
                det = det[keep]

                preds.append({
                    'boxes': det[:, :4] if det.numel() else torch.zeros((0, 4), dtype=torch.float32),
                    'scores': det[:, 4] if det.numel() else torch.zeros((0,), dtype=torch.float32),
                    'labels': det[:, 5].to(torch.int64) if det.numel() else torch.zeros((0,), dtype=torch.int64),
                })
                refs.append({
                    'boxes': target['bbox'].detach().cpu(),
                    'labels': target['cls'].detach().cpu(),
                })

            metric.update(preds, refs)

    computed = metric.compute()
    return {
        'map': float(computed['map']),
        'map_50': float(computed['map_50']),
        'mar_100': float(computed['mar_100']),
    }


for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_metrics = evaluate_model(model, val_loader, DEVICE, SCORE_THRESHOLD)
    lr_scheduler.step()

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'map': val_metrics['map'],
        'map_50': val_metrics['map_50'],
        'mar_100': val_metrics['mar_100'],
        'lr': optimizer.param_groups[0]['lr'],
    }
    history.append(row)
    print(row)

    torch.save(model.state_dict(), last_weights_path)
    if row['map_50'] > best_map50:
        best_map50 = row['map_50']
        torch.save(model.state_dict(), best_weights_path)

history_df = pd.DataFrame(history)
history_path = RUNS_DIR / 'metrics.csv'
history_df.to_csv(history_path, index=False)
history_df


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_df['epoch'], history_df['train_loss'], marker='o')
plt.title('Train Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_df['epoch'], history_df['map_50'], marker='o', label='mAP@50')
plt.plot(history_df['epoch'], history_df['map'], marker='o', label='mAP@50:95')
plt.title('Validation Metrics')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.legend()
plt.grid(True)
plt.tight_layout()

metrics_plot_path = RUNS_DIR / 'training_curves.png'
plt.savefig(metrics_plot_path, dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torchvision.transforms import functional as F
from effdet import create_model

best_model = create_model('tf_efficientdet_d0', bench_task='predict', num_classes=NUM_CLASSES)
best_model.load_state_dict(torch.load(best_weights_path, map_location=DEVICE))
best_model.to(DEVICE)
best_model.eval()

sample_count = 6
base_val_dataset = val_dataset.dataset if hasattr(val_dataset, 'dataset') else val_dataset
subset_indices = val_dataset.indices if hasattr(val_dataset, 'indices') else list(range(len(base_val_dataset)))
selected_indices = subset_indices[:min(sample_count, len(subset_indices))]

preview_dir = PREDICTIONS_DIR / 'val_preview'
preview_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(14, 10))
with torch.no_grad():
    for plot_idx, dataset_idx in enumerate(selected_indices, start=1):
        record = base_val_dataset.records[dataset_idx]
        image_path = record['image_path']
        image = Image.open(image_path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
        image_tensor = F.to_tensor(image).unsqueeze(0).to(DEVICE)
        outputs = best_model(image_tensor)

        if isinstance(outputs, dict):
            detections = outputs.get('detections', None)
        else:
            detections = getattr(outputs, 'detections', None)
        det = detections[0].detach().cpu() if detections is not None else torch.zeros((0, 6))
        keep = det[:, 4] >= SCORE_THRESHOLD if det.numel() else torch.zeros((0,), dtype=torch.bool)
        det = det[keep] if det.numel() else det

        image_np = np.array(image).copy()
        for row in det:
            x1, y1, x2, y2, score, label = row.tolist()
            x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
            label = int(label)
            class_name = CLASS_NAMES[label - 1] if 1 <= label <= len(CLASS_NAMES) else str(label)
            cv2.rectangle(image_np, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(image_np, f'{class_name} {score:.2f}', (x1, max(20, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        output_path = preview_dir / f'image{plot_idx - 1}.jpg'
        Image.fromarray(image_np).save(output_path)

        plt.subplot(2, 3, plot_idx)
        plt.imshow(image_np)
        plt.axis('off')
        plt.title(image_path.name)

plt.tight_layout()
plt.show()


In [ ]:
import shutil
import json

final_dir = ARTIFACTS_DIR / 'final_artifacts'
final_dir.mkdir(parents=True, exist_ok=True)

if len(history_df) > 0:
    best_row_idx = history_df['map_50'].idxmax()
    summary_metrics = history_df.loc[best_row_idx].to_dict()
else:
    summary_metrics = {}

summary_metrics_path = final_dir / 'best_metrics.json'
summary_metrics_path.write_text(json.dumps(summary_metrics, ensure_ascii=False, indent=2), encoding='utf-8')

files_to_copy = [
    best_weights_path,
    last_weights_path,
    history_path,
    metrics_plot_path,
    summary_path,
]

for file_path in files_to_copy:
    if file_path.exists():
        destination = final_dir / file_path.name
        if file_path.resolve() != destination.resolve():
            shutil.copy2(file_path, destination)

preview_dir = PREDICTIONS_DIR / 'val_preview'
final_preview_dir = final_dir / 'val_preview'

if preview_dir.exists():
    if final_preview_dir.exists():
        shutil.rmtree(final_preview_dir)
    shutil.copytree(preview_dir, final_preview_dir)

print('Artifacts directory:', final_dir)
print('Available files:')
for path in sorted(final_dir.glob('*')):
    print('-', path.name)


## ??? ????????? ????? ????????

????? ?????????? ???????? ??????? ????????? `best_efficientdet.pt`, `last_efficientdet.pt`, `metrics.csv`, `training_curves.png`, `dataset_summary.json` ? `best_metrics.json`.
